In [2]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_1.xlsx"

try:
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_series = None

    # We iterate through rows while keeping track of the cell objects
    for row in ws.iter_rows(min_row=2):
        # Extract values for logic
        values = [cell.value for cell in row if cell.value not in (None, "Grand Total")]
        
        # We need the first cell object specifically to check for Bold formatting
        first_cell = row[0]
        is_bold = first_cell.font.bold if first_cell.font else False
        
        # Extract numeric values
        nums = [v for v in values if isinstance(v, (int, float))]
        texts = [str(v).strip() for v in values if isinstance(v, str)]

        if not texts or not nums:
            continue

        label = texts[0]
        qty = int(nums[-1])

        # NEW LOGIC: If it's bold OR qty > 1, it's a parent
        if is_bold or qty > 1:
            current_series = label
        
        # If it's NOT bold AND qty == 1, it's a child part
        elif qty == 1 and current_series:
            rows.append({
                "Series": current_series,
                "Part": label, 
                "Count": 1
            })

    df = pd.DataFrame(rows)
    
    if not df.empty:
        df.to_excel(output_file, index=False)
        print(f"Success! Processed {len(df)} parts into their respective series.")
    else:
        print("Processing complete, but no child parts were mapped.")

except Exception as e:
    print(f"An error occurred: {e}")

new wxcel file created 114351 rows


In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

try:
    print("Loading workbook... (Large file, please wait)")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_parent = None
    child_remaining = 0

    print("Processing rows...")
    # Iterating through rows
    for row in ws.iter_rows(min_row=2):
        # Identify key cells (Adjust index if Label is not Col A and Qty is not Col B)
        label_cell = row[0]
        qty_cell = row[1]
        
        label = str(label_cell.value).strip() if label_cell.value is not None else ""
        if not label or "Grand Total" in label:
            continue

        # Check for Bold (Parent indicator)
        is_bold = label_cell.font.bold if label_cell.font else False
        
        try:
            qty = int(qty_cell.value) if qty_cell.value is not None else 0
        except (ValueError, TypeError):
            qty = 0

        # LOGIC ENGINE:
        if is_bold:
            # This is a Parent. Update the current parent and set the countdown.
            current_parent = label
            child_remaining = qty
            # We don't add the parent to the 'Part' list, just store it as the header.
            continue 
        
        elif child_remaining > 0:
            # This is a Child. Map it to the active parent and decrement the count.
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Status": "Mapped"
            })
            child_remaining -= 1
            
        else:
            # This handles rows that are neither bold parents nor within a parent's count
            # Helpful for debugging data gaps
            if label:
                rows.append({
                    "Parent Series": "UNASSOCIATED",
                    "Child Part": label,
                    "Status": "Review Needed"
                })

    # Exporting
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    
    print(f"Done! Processed {len(df)} associations.")
    if child_remaining > 0:
        print(f"Warning: The last parent expected {child_remaining} more children than were found.")

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final.xlsx"

def has_bold_cell(row):
    """Checks if any cell in the row has bold formatting."""
    for cell in row:
        if cell.font and cell.font.bold:
            return True
    return False

try:
    print("Loading workbook...")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_parent = "Unknown/Start"
    child_remaining = 0

    print("Processing rows...")
    # Using min_row=2 to skip headers
    for i, row in enumerate(ws.iter_rows(min_row=2), start=2):
        # 1. Extract values
        label = str(row[0].value).strip() if row[0].value is not None else ""
        
        # Get Qty (assuming it's in Column B / index 1)
        try:
            qty_val = row[1].value
            qty = int(float(qty_val)) if qty_val is not None else 0
        except (ValueError, TypeError):
            qty = 0

        if not label or "TOTAL" in label.upper():
            continue

        # 2. Stronger Parent Detection
        # A row is a parent if it's BOLD or if it has a quantity but we aren't currently filling a parent's quota
        is_bold = has_bold_cell(row)
        
        if is_bold and qty > 0:
            current_parent = label
            child_remaining = qty
            continue # Move to next row to start collecting children
        
        # 3. Association Logic
        if child_remaining > 0:
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Child_Qty_Check": 1
            })
            child_remaining -= 1
        else:
            # This logic captures parents that might not be bold but have children
            if qty > 0:
                current_parent = label
                child_remaining = qty
            else:
                rows.append({
                    "Parent Series": "ORPHAN",
                    "Child Part": label,
                    "Child_Qty_Check": 0
                })

    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"Done! Created {output_file} with {len(df)} rows.")

except Exception as e:
    print(f"Error occurred: {e}")

In [ ]:
# alert !!!!!!!!!!!!!!!!!!


# havent used yet

from openpyxl import load_workbook
import pandas as pd

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
wb = load_workbook(input_file, data_only=True)
ws = wb.active

rows = []

current_parent = None
remaining_children = 0

for row in ws.iter_rows(min_row=2):

    cell_text = row[0].value
    cell_qty  = row[-1].value

    if cell_text is None or cell_qty is None:
        continue

    # Detect bold row → parent
    if row[0].font.bold:
        current_parent = str(cell_text).strip()
        remaining_children = int(cell_qty)
        continue

    # Assign child parts
    if current_parent and remaining_children > 0:
        rows.append({
            "Parent": current_parent,
            "Child": str(cell_text).strip(),
            "count": 1
        })
        remaining_children -= 1

df = pd.DataFrame(rows)

output_file = "parent_child_mapping.xlsx"
df.to_excel(output_file, index=False)

print("Excel created with", len(df), "child rows")


In [ ]:
import pandas as pd

# Path to your file
file_path = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"

print("Loading data...")
# Read the file (adjust sheet_name if necessary)
df = pd.read_excel(file_path)

# 1. Count of exact duplicate rows (where every column is identical)
duplicate_count = df.duplicated().sum()

# 2. To see the actual duplicate rows:
duplicates = df[df.duplicated(keep=False)]

print(f"Total Rows: {len(df)}")
print(f"Total Duplicate Rows: {duplicate_count}")

if duplicate_count > 0:
    # Optional: Save duplicates to a file to inspect them
    duplicates.to_excel("duplicate_rows_found.xlsx", index=False)
    print("A file 'duplicate_rows_found.xlsx' has been created for your review.")
else:
    print("Every row in the file is unique.")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_final_tallied.xlsx"

try:
    print("Loading workbook... (114k rows may take a moment)")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_parent = None
    target_sum = 0
    running_sum = 0

    # Iterate through all rows
    for i, row in enumerate(ws.iter_rows(min_row=1), start=1):
        # Mapping: Col A = Label, Col C = Qty
        label_cell = row[0]
        qty_cell = row[2] # Column C

        label = str(label_cell.value).strip() if label_cell.value is not None else ""
        
        try:
            # Using float first to handle decimals, then converting for logic
            qty = float(qty_cell.value) if qty_cell.value is not None else 0
        except (ValueError, TypeError):
            qty = 0

        # Skip rows that have absolutely no data
        if not label and qty == 0:
            continue

        # Logic 1: Identify Parent (Must be BOLD)
        is_bold = label_cell.font.bold if label_cell.font else False

        if is_bold:
            # If we were already tracking a parent but hit a new bold row 
            # before the sum was met, it's a data anomaly we should catch.
            if current_parent and running_sum != target_sum:
                print(f"Warning: Parent '{current_parent}' ended early at row {i-1}. "
                      f"Expected {target_sum}, found {running_sum}")

            current_parent = label
            target_sum = qty
            running_sum = 0
            continue 
        
        # Logic 2: Identify Children
        elif current_parent:
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Child Qty": qty,
                "Group Target": target_sum
            })
            running_sum += qty
            
            # Close the group once the math matches
            if running_sum >= target_sum:
                current_parent = None
                target_sum = 0
                running_sum = 0
        
        # Logic 3: Handle Unassociated rows
        else:
            if label:
                rows.append({
                    "Parent Series": "MANUAL REVIEW REQUIRED",
                    "Child Part": label,
                    "Child Qty": qty,
                    "Group Target": 0
                })

    # Final Export
    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"\nProcessing Complete!")
    print(f"Total entries mapped: {len(df)}")
    print(f"Output saved to: {output_file}")

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import pandas as pd
from openpyxl import load_workbook

input_file = "C:/Users/Ex0164/Tushar vats/Copy of 1401 BOM.xlsx"
output_file = "series_part_mapping_2col_tallied.xlsx"

try:
    print("Loading workbook... (114k rows may take a moment)")
    wb = load_workbook(input_file, data_only=True)
    ws = wb.active
    
    rows = []
    current_parent = None
    target_sum = 0
    running_sum = 0

    print("Processing 2-column structure...")
    for i, row in enumerate(ws.iter_rows(min_row=1), start=1):
        # Ensure the row has at least 2 cells to avoid 'index out of range'
        if len(row) < 2:
            continue 

        # Mapping: Col A = Label (0), Col B = Qty (1)
        label_cell = row[0]
        qty_cell = row[1] 

        label = str(label_cell.value).strip() if label_cell.value is not None else ""
        
        try:
            qty = float(qty_cell.value) if qty_cell.value is not None else 0
        except (ValueError, TypeError):
            qty = 0

        if not label and qty == 0:
            continue

        # Logic 1: Identify Parent (Must be BOLD)
        is_bold = label_cell.font.bold if label_cell.font else False

        if is_bold:
            # Check for early exit anomalies
            if current_parent and running_sum != target_sum:
                print(f"Warning: Parent '{current_parent}' ended early at row {i-1}. "
                      f"Expected {target_sum}, found {running_sum}")

            current_parent = label
            target_sum = qty
            running_sum = 0
            continue 
        
        # Logic 2: Identify Children
        elif current_parent:
            rows.append({
                "Parent Series": current_parent,
                "Child Part": label,
                "Child Qty": qty,
                "Group Target": target_sum
            })
            running_sum += qty
            
            # Reset parent once the tally is reached
            if running_sum >= target_sum:
                current_parent = None
                target_sum = 0
                running_sum = 0
        
        # Logic 3: Handle Unassociated rows
        else:
            if label:
                rows.append({
                    "Parent Series": "UNASSOCIATED",
                    "Child Part": label,
                    "Child Qty": qty
                })

    df = pd.DataFrame(rows)
    df.to_excel(output_file, index=False)
    print(f"\nProcessing Complete! File saved as: {output_file}")

except Exception as e:
    print(f"An error occurred: {e}")